# ByT5 Fine-tuning untuk Silabisasi Pantun (Colab)

Notebook ini melatih model seq2seq ByT5 dari CSV di Google Drive.

Aturan target:
- pakai kolom anotasi (`annotated` / `anotated` / `anotasi`) jika terisi
- jika kosong, fallback ke kolom `silabel`


In [ ]:
!pip -q install transformers datasets sentencepiece accelerate evaluate sacrebleu pandas

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

MODEL_NAME = 'google/byt5-small'
DATA_CSV = '/content/drive/MyDrive/pantun_byt5.csv'  # ganti bila perlu
OUTPUT_DIR = '/content/drive/MyDrive/Models/byt5-silabel'

MAX_SOURCE_LENGTH = 128
MAX_TARGET_LENGTH = 128
TRAIN_BATCH_SIZE = 8
EVAL_BATCH_SIZE = 8
EPOCHS = 8
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.1
MAX_GRAD_NORM = 1.0
GRADIENT_ACCUMULATION_STEPS = 2
USE_FP16 = False  # set True kalau sudah stabil
TASK_PREFIX = 'silabisasi: '
SEED = 42

In [ ]:
import pandas as pd

df = pd.read_csv(DATA_CSV)
if 'text' not in df.columns:
    raise ValueError("CSV wajib punya kolom 'text'.")
if 'silabel' not in df.columns:
    raise ValueError("CSV wajib punya kolom 'silabel'.")

annot_candidates = ['annotated', 'anotated', 'anotasi']
annot_col = next((c for c in annot_candidates if c in df.columns), None)
if annot_col is None:
    print("Kolom anotasi tidak ditemukan; semua target pakai 'silabel'.")
    annot_values = pd.Series([''] * len(df), index=df.index)
else:
    annot_values = df[annot_col].fillna('').astype(str).str.strip()

df['text'] = df['text'].fillna('').astype(str).str.strip()
df['silabel'] = df['silabel'].fillna('').astype(str).str.strip()
df['target'] = annot_values.where(annot_values != '', df['silabel'])

before = len(df)
df = df[(df['text'] != '') & (df['target'] != '')].copy()
df = df[['text', 'target']].drop_duplicates().reset_index(drop=True)

fallback_count = int((annot_values == '').sum()) if annot_col else len(df)
print(f'Jumlah baris awal        : {before}')
print(f'Jumlah baris siap latih  : {len(df)}')
print(f'Fallback ke silabel      : {fallback_count}')
if annot_col:
    print(f'Kolom anotasi terdeteksi : {annot_col}')
display(df.head(10))

In [ ]:
from datasets import Dataset, DatasetDict

dataset = Dataset.from_pandas(df)
split_test = dataset.train_test_split(test_size=0.1, seed=SEED)
split_train_val = split_test['train'].train_test_split(test_size=0.111111, seed=SEED)

ds = DatasetDict({
    'train': split_train_val['train'],
    'validation': split_train_val['test'],
    'test': split_test['test'],
})

print(ds)

In [ ]:
import numpy as np
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def preprocess(batch):
    source_texts = [TASK_PREFIX + t for t in batch['text']]
    model_inputs = tokenizer(
        source_texts,
        max_length=MAX_SOURCE_LENGTH,
        truncation=True,
    )
    labels = tokenizer(
        text_target=batch['target'],
        max_length=MAX_TARGET_LENGTH,
        truncation=True,
    )
    model_inputs['labels'] = labels['input_ids']
    return model_inputs

tokenized_ds = ds.map(preprocess, batched=True, remove_columns=ds['train'].column_names)
tokenized_ds

In [ ]:
import inspect
import torch
from transformers import (
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    EarlyStoppingCallback,
)

model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)

def _to_valid_token_ids(arr):
    if isinstance(arr, tuple):
        arr = arr[0]
    arr = np.asarray(arr)
    if arr.ndim == 3:
        arr = np.argmax(arr, axis=-1)
    arr = arr.astype(np.int64, copy=False)
    arr = np.where(arr == -100, tokenizer.pad_token_id, arr)
    arr = np.where(arr < 0, tokenizer.pad_token_id, arr)
    max_id = len(tokenizer) - 1
    arr = np.where(arr > max_id, tokenizer.pad_token_id, arr)
    return arr

def compute_metrics(eval_pred):
    preds, labels = eval_pred
    pred_ids = _to_valid_token_ids(preds)
    label_ids = _to_valid_token_ids(labels)
    decoded_preds = tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(label_ids, skip_special_tokens=True)

    decoded_preds = [p.strip() for p in decoded_preds]
    decoded_labels = [l.strip() for l in decoded_labels]
    exact = np.mean([int(p == l) for p, l in zip(decoded_preds, decoded_labels)])
    return {'exact_match': float(exact)}

arg_sig = inspect.signature(Seq2SeqTrainingArguments.__init__).parameters
raw_args_kwargs = {
    'output_dir': OUTPUT_DIR,
    'logging_steps': 50,
    'learning_rate': LEARNING_RATE,
    'per_device_train_batch_size': TRAIN_BATCH_SIZE,
    'per_device_eval_batch_size': EVAL_BATCH_SIZE,
    'gradient_accumulation_steps': GRADIENT_ACCUMULATION_STEPS,
    'weight_decay': WEIGHT_DECAY,
    'max_grad_norm': MAX_GRAD_NORM,
    'num_train_epochs': EPOCHS,
    'predict_with_generate': True,
    'generation_max_length': MAX_TARGET_LENGTH,
    'fp16': USE_FP16 and torch.cuda.is_available(),
    'load_best_model_at_end': True,
    'metric_for_best_model': 'eval_exact_match',
    'greater_is_better': True,
    'save_total_limit': 2,
    'seed': SEED,
}
args_kwargs = {k: v for k, v in raw_args_kwargs.items() if k in arg_sig}

if 'eval_strategy' in arg_sig:
    args_kwargs['eval_strategy'] = 'epoch'
elif 'evaluation_strategy' in arg_sig:
    args_kwargs['evaluation_strategy'] = 'epoch'

if 'save_strategy' in arg_sig:
    args_kwargs['save_strategy'] = 'epoch'
if 'logging_strategy' in arg_sig:
    args_kwargs['logging_strategy'] = 'steps'
if 'warmup_ratio' in arg_sig:
    args_kwargs['warmup_ratio'] = WARMUP_RATIO
if 'generation_num_beams' in arg_sig:
    args_kwargs['generation_num_beams'] = 4

args = Seq2SeqTrainingArguments(**args_kwargs)

trainer_kwargs = {
    'model': model,
    'args': args,
    'train_dataset': tokenized_ds['train'],
    'eval_dataset': tokenized_ds['validation'],
    'data_collator': collator,
    'compute_metrics': compute_metrics,
    'callbacks': [EarlyStoppingCallback(early_stopping_patience=3)],
}
if 'processing_class' in inspect.signature(Seq2SeqTrainer.__init__).parameters:
    trainer_kwargs['processing_class'] = tokenizer
else:
    trainer_kwargs['tokenizer'] = tokenizer

trainer = Seq2SeqTrainer(**trainer_kwargs)

trainer.train()
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f'Model tersimpan di: {OUTPUT_DIR}')

In [ ]:
test_metrics = trainer.evaluate(tokenized_ds['test'], metric_key_prefix='test')
print(test_metrics)

pred_out = trainer.predict(tokenized_ds['test'])
pred_ids = _to_valid_token_ids(pred_out.predictions)
label_ids = _to_valid_token_ids(pred_out.label_ids)
pred_texts = tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
label_texts = tokenizer.batch_decode(label_ids, skip_special_tokens=True)

preview = pd.DataFrame({
    'text': ds['test']['text'][:20],
    'gold': [x.strip() for x in label_texts[:20]],
    'pred': [x.strip() for x in pred_texts[:20]],
})
display(preview)

In [ ]:
def predict_silabel(text: str) -> str:
    prefixed = TASK_PREFIX + text.strip()
    inputs = tokenizer(prefixed, return_tensors='pt').to(model.device)
    model.eval()
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=MAX_TARGET_LENGTH,
            num_beams=4,
            no_repeat_ngram_size=3,
            repetition_penalty=1.2,
            early_stopping=True,
        )
    return tokenizer.decode(output_ids[0], skip_special_tokens=True).strip()

print(predict_silabel('menyukai sifatmu yang ramah'))